In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
SOURCE_TABLE = "gs_carbon.bronze.openmeteo_weather"
TARGET_TABLE = "gs_carbon.silver.openmeteo_weather"

In [0]:
df = spark.table(SOURCE_TABLE)

df = (
    df
    .withColumn(
        "year",
        F.year("reference_date")
    )
    .withColumn(
        "month",
        F.month("reference_date")
    )
    .withColumn(
        "year_month",
        F.date_format(
            "reference_date",
            "yyyy-MM"
        )
    )
    .withColumn(
        "quarter",
        F.quarter("reference_date")
    )
)

df = (
    df
    .withColumn(
        "temperature_category",
        F.when(
            F.col("temperature_mean") <= 15,
            "Frio"
        )
        .when(
            F.col("temperature_mean") <= 25,
            "Ameno"
        )
        .otherwise(
            "Quente"
        )
    )
)

df = (
    df
    .withColumn(
        "rain_category",
        F.when(
            F.col("precipitation_sum") == 0,
            "Sem chuva"
        )
        .when(
            F.col("precipitation_sum") <= 10,
            "Baixa"
        )
        .when(
            F.col("precipitation_sum") <= 50,
            "Moderada"
        )
        .otherwise(
            "Alta"
        )
    )
)

spark.sql("""
CREATE SCHEMA IF NOT EXISTS gs_carbon.silver
""")

(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema","true")
      .saveAsTable(TARGET_TABLE)
)

print(
    f"Silver atualizada: {TARGET_TABLE}"
)

In [0]:
%sql
SELECT * FROM gs_carbon.silver.openmeteo_weather